# Assignment 1: Gaussians, Categories, and Clusters

**This is the Python (no-GenJAX) stencil.** Problems are solved with `numpy` + `scipy.stats` + `matplotlib`.

**Optional**: each numpy code cell is followed by a paired *"Now in GenJAX"* cell. These cells are not required, but they walk you through doing the same task using GenJAX, with enough inline explanation that you can complete them without prior GenJAX experience. They are a good on-ramp if you have not done the Tutorial 2 Ch 0–4 readings.

**Corresponding textbook chapters:** Tutorial 1 Ch 5 (Bayesian inference) and [Tutorial 2 Ch 5 — Mixture Models](https://josephausterweil.github.io/probintro/intro2/05_mixture_models/).


## Setup

Run the cells below to install (if needed) and import everything. The GenJAX install only matters if you plan to do the optional GenJAX cells.


In [ ]:
# Optional: only needed if you do the "Now in GenJAX" cells.
# In Google Colab, uncomment the line below on first run:
# !pip install genjax


In [ ]:
# Standard imports for the numpy-only path.
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

np.random.seed(42)

# GenJAX imports (only needed for the optional "Now in GenJAX" cells).
# If you skip those cells, you can ignore an ImportError here.
try:
    import jax
    import jax.numpy as jnp
    import jax.random as random
    from genjax import gen, normal, bernoulli
    key = random.PRNGKey(42)
    _GENJAX_AVAILABLE = True
except ImportError:
    _GENJAX_AVAILABLE = False
    print("GenJAX not available — the 'Now in GenJAX' cells will not run. The numpy path works fine without it.")


---

# Problem 1: Gaussian-Gaussian Conjugate Model

We start with the **Gaussian-Gaussian** conjugate model: a Gaussian likelihood with unknown mean $\mu$ and known variance $\sigma_x^2$, with a Gaussian prior on $\mu$:

$$\mu \sim \mathcal{N}(\mu_0, \sigma_0^2) \qquad \qquad x_1, \dotsc, x_N \mid \mu, \sigma_x^2 \overset{iid}{\sim} \mathcal{N}(\mu, \sigma_x^2)$$

The conjugate posterior and posterior-predictive are:

$$\mu \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\frac{\mu_0 \sigma_0^{-2} + \sigma_x^{-2} \sum_n x_n}{\sigma_0^{-2} + N \sigma_x^{-2}},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1}\right)$$

$$x_{N+1} \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\text{same mean as posterior},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1} + \sigma_x^2\right)$$

**For Problem 1, use $\mu_0 = 0$ and $\sigma_0^2 = 1$.**


## Part 1(a): Prior plot

To provide a baseline, plot the prior distribution $p(\mu) = \mathcal{N}(\mu; \mu_0, \sigma_0^2)$ over a range that captures both tails and the peak.


In [ ]:
# fill me
#
# Suggested approach:
#   1. mu_0 = 0, sigma_0_squared = 1.
#   2. mu_range = np.linspace(-4, 4, 1000).
#   3. Evaluate the prior density with norm.pdf(mu_range, mu_0, np.sqrt(sigma_0_squared)).
#   4. Plot density vs. mu_range; label the axes.

mu_0 = 0.0
sigma_0_squared = 1.0
mu_range = np.linspace(-4, 4, 1000)

fig, ax = plt.subplots(figsize=(8, 5))

# your plotting code here

ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$p(\mu)$')
ax.set_title(r'Prior: $\mathcal{N}(\mu_0, \sigma_0^2)$')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Now in GenJAX — Part 1(a): the prior as a generative function

**Concept (Tutorial 2 Ch 2): `@gen` functions.** A GenJAX **generative function** is a Python function decorated with `@gen` that defines a *probabilistic process*. Each call to `simulate` runs the function once with a fresh random key and returns a **trace** — a record of the random choices that were made.

Below, you'll write the prior as a `@gen` function and use it to sample many values of $\mu$. Then you'll histogram the samples and overlay the analytical density. They should match (this is "Monte Carlo as sanity check").

**Key syntax:**
- `mu = normal(mu_0, sigma_0) @ "mu"` samples a normal RV and **addresses** the choice with the name `"mu"`. You can later refer to this choice by name.
- `prior.simulate(key, args)` runs the function once; `prior.simulate.vmap(in_axes=...)(keys, args)` runs it in parallel for many keys (see Ch 2).
- `trace.get_retval()` returns whatever the `@gen` function returned.

**Why bother?** For a conjugate Gaussian prior, plotting the analytical density is faster. But Monte Carlo sampling is what you'll *have* to do once the model becomes non-conjugate (Problem 2 Part (e), or any real research problem). This cell teaches the pattern.


In [ ]:
# fill me — optional GenJAX path
#
# Suggested approach:
#   1. Write a @gen function `prior(mu_0, sigma_0)` that samples mu from normal(mu_0, sigma_0)
#      addressed as "mu" and returns it. NOTE: GenJAX `normal` takes (mean, std), not (mean, var).
#   2. Vectorize: keys = random.split(key, N) with N = 5000.
#   3. Run `traces = jax.vmap(lambda k: prior.simulate(k, (mu_0, sigma_0)))(keys)`.
#   4. Pull out the samples with `mu_samples = traces.get_retval()`.
#   5. Plot histogram (density=True) over mu_range and overlay norm.pdf(mu_range, mu_0, sigma_0).

if _GENJAX_AVAILABLE:
    mu_0 = 0.0
    sigma_0 = 1.0  # std, not variance, for GenJAX `normal`
    N = 5000

    @gen
    def prior(mu_0, sigma_0):
        # mu = normal(mu_0, sigma_0) @ "mu"
        # return mu
        pass

    # your sampling + plotting code here
else:
    print("GenJAX not installed — skipping.")


## Part 1(b): One-datum update

Calculate and plot the **posterior** $p(\mu \mid x_1)$ **and** the **posterior-predictive** $p(x_2 \mid x_1)$ after observing $x_1 = 2$, for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$ (four distributions total).

**Question.** How does changing the likelihood variance $\sigma_x^2$ affect the posterior and the predictive? Where are the two distributions similar? Where do they differ, and why?


In [ ]:
def conjugate_update(mu_0, sigma_0_squared, sigma_x_squared, data):
    """
    Conjugate Gaussian-Gaussian update.

    Returns (post_mean, post_var, pred_var).
    post_var is the posterior variance of mu; pred_var = post_var + sigma_x_squared.
    """
    # fill me
    #
    # 1. N = len(data); sum_x = data.sum() (handle N=0 as the prior).
    # 2. posterior_precision = 1/sigma_0_squared + N / sigma_x_squared.
    # 3. post_mean = (mu_0 / sigma_0_squared + sum_x / sigma_x_squared) / posterior_precision.
    # 4. post_var  = 1 / posterior_precision.
    # 5. pred_var  = post_var + sigma_x_squared.
    pass


In [ ]:
# fill me
#
# Plot p(mu | x_1=2) and p(x_2 | x_1=2) for sigma_x_squared in {0.25, 4}.

mu_0 = 0.0
sigma_0_squared = 1.0
x_1 = np.array([2.0])
sigma_x_squared_values = [0.25, 4.0]
plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    # 1. post_mean, post_var, pred_var = conjugate_update(...)
    # 2. Plot posterior  norm.pdf(plot_range, post_mean, np.sqrt(post_var)).
    # 3. Plot predictive norm.pdf(plot_range, post_mean, np.sqrt(pred_var)).
    # 4. Mark x_1 with a vertical line.

    # your code here

    ax.set_xlabel('value')
    ax.set_ylabel('density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared}$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


### Now in GenJAX — Part 1(b): conditioning with `generate`

**Concept (Tutorial 2 Ch 4): conditioning a `@gen` function.** GenJAX gives you the posterior by *conditioning* the model on observed data. There are three ways:

1. **Rejection sampling** — simulate many traces, throw away the ones inconsistent with the observation. Simple to understand; very inefficient for continuous observations (probability of exact match = 0).
2. **`model.generate(key, constraints, args)`** — runs the model but **forces** addressed choices to take observed values, returning the trace + an importance weight.
3. **Importance sampling** — for posterior over an unobserved latent: run `generate` many times and weight each sample by its importance score.

For this cell we'll use **option 3** because the latent ($\mu$) is continuous and we want a posterior over it.

**Why bother?** For conjugate Gaussian-Gaussian, you already have the closed-form posterior — importance sampling is overkill. But this is the *exact* pattern that scales to non-conjugate models where no closed form exists. Doing it here on a problem with a known answer means you can *verify* your importance-sampling code by comparing the weighted histogram to the analytical posterior.

**Key syntax:**
- A **choice map** `{"x_1": 2.0}` tells `generate` to constrain the choice addressed `"x_1"` to the value `2.0`.
- `trace, weight = model.generate(key, ChoiceMap({"x_1": 2.0}), args)` returns the trace and the log-importance-weight.
- A weighted histogram with `plt.hist(mu_samples, weights=np.exp(log_weights - log_weights.max()), density=True)` gives the posterior over $\mu$.


In [ ]:
# fill me — optional GenJAX path
#
# Suggested approach:
#   1. Finish the @gen `gaussian_gaussian_model` below (uncomment the body lines).
#      NOTE: GenJAX `normal` takes (mean, std), not (mean, var).
#   2. For each sigma_x_squared in {0.25, 4}:
#        - Build constraints = ChoiceMap({"x_1": 2.0}).
#        - Importance-sample: for N keys, call gaussian_gaussian_model.generate(key, constraints, args).
#        - Extract mu_samples = traces.get_choices()["mu"] and log_weights = importance_weights.
#        - Plot a weighted histogram of mu_samples; overlay the closed-form posterior from above.
#      Compare — they should match.

# GenJAX importance-sampling API note:
# from genjax import ChoiceMap   # or build via {"x_1": 2.0} depending on your GenJAX version
# Check Tutorial 2 Ch 4 for the exact ChoiceMap construction in your version.

if _GENJAX_AVAILABLE:
    mu_0 = 0.0
    sigma_0 = 1.0  # std

    @gen
    def gaussian_gaussian_model(mu_0, sigma_0, sigma_x):
        # mu = normal(mu_0, sigma_0) @ "mu"
        # x_1 = normal(mu, sigma_x) @ "x_1"
        # return mu, x_1
        pass

    # For each sigma_x_squared in {0.25, 4}, condition on x_1 = 2 and importance-sample.
    # your code here
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 1(b)).**

- Where are the posterior and predictive distributions *similar*, and where do they *differ*?
- What is the effect of increasing $\sigma_x^2$ from 0.25 to 4? Which distribution moves more? Why?

*(your answer here)*


## Part 1(c): Multiple-datum update

Now observe five data points: $(x_1, \dotsc, x_5) = (2.1, 2.5, 1.4, 2.2, 1.8)$. Plot the posterior and predictive for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$.

The average of these five points is exactly $2.0$, same as the single datum in Part 1(b). **Compare** the resulting posteriors and predictives to Part 1(b).


In [ ]:
# fill me
#
# Reuse conjugate_update with the 5-point data array.

mu_0 = 0.0
sigma_0_squared = 1.0
data = np.array([2.1, 2.5, 1.4, 2.2, 1.8])
sigma_x_squared_values = [0.25, 4.0]
plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    # Same recipe as Part 1(b), but with the 5-point data array.

    # your code here

    ax.set_xlabel('value')
    ax.set_ylabel('density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared},\ N = 5$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


### Now in GenJAX — Part 1(c): conditioning on multiple observations

**Concept extension (Tutorial 2 Ch 4): multiple observations in one choice map.** The same `generate` API handles many observations at once — just put them all into the choice map.

**Two ways to express N observations:**

1. **Plate notation in the model.** Inside the `@gen` function, sample N observations addressed as `"x_1"`, `"x_2"`, ..., `"x_N"`. Then your choice map is `{"x_1": 2.1, "x_2": 2.5, ..., "x_5": 1.8}`. Easy to read; verbose.

2. **Vectorized addressing (Tutorial 2 Ch 6).** Use `jax.vmap` over a single `"x"` address so the model produces an array of observations under one address. More concise; requires understanding `vmap`.

For this cell, **use approach 1** (it's clearer when you're first learning the pattern). Approach 2 is a Tutorial 2 Ch 6 topic.


In [ ]:
# fill me — optional GenJAX path
#
# Suggested approach (approach 1: explicit addresses):
#   1. Finish the @gen `gaussian_gaussian_5` below — add lines for x_1 through x_5,
#      each addressed as "x_1", "x_2", ..., "x_5".
#   2. Constraints = ChoiceMap({"x_1": 2.1, "x_2": 2.5, "x_3": 1.4, "x_4": 2.2, "x_5": 1.8}).
#   3. Importance-sample as in Part 1(b), now compare to the N=5 closed-form posterior.

if _GENJAX_AVAILABLE:
    mu_0 = 0.0
    sigma_0 = 1.0
    data_5 = [2.1, 2.5, 1.4, 2.2, 1.8]

    @gen
    def gaussian_gaussian_5(mu_0, sigma_0, sigma_x):
        # mu = normal(mu_0, sigma_0) @ "mu"
        # x_1 = normal(mu, sigma_x) @ "x_1"
        # x_2 = normal(mu, sigma_x) @ "x_2"
        # x_3 = normal(mu, sigma_x) @ "x_3"
        # x_4 = normal(mu, sigma_x) @ "x_4"
        # x_5 = normal(mu, sigma_x) @ "x_5"
        # return mu
        pass

    # For each sigma_x_squared in {0.25, 4}, condition on all 5 observations and importance-sample.
    # your code here
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 1(c)).**

- How do the Part 1(c) posteriors compare to Part 1(b)? Where are they the same? Where do they differ?
- The data average is 2.0 in both cases, so the posterior *mean* should be similar. What about the posterior *variance*?
- What about the predictive variance? Is it dominated by the posterior variance or by $\sigma_x^2$?

*(your answer here)*


---

# Problem 2: Gaussian Mixture / Categorization

In this problem, we make **categorization decisions** for two categories, each defined as a Gaussian distribution. You will derive the probability of an item being in one category vs. the other, then explore how the variances and prior probability of each category affect the posterior and the predictive distribution.

Data are generated by first picking which of two categories $c = 1, 2$ a datum belongs to (according to their prior probability) and then generating the datum from the corresponding category's likelihood:

$$c_n | \theta \sim \text{Bernoulli}(\theta) \qquad \qquad x_n | \mu_{c(n)}, \sigma_{c(n)}^2 \overset{iid}{\sim} \mathcal{N}(\mu_{c(n)}, \sigma_{c(n)}^2)$$

$c_n = 1$ with probability $\theta$ (and $c_n = 2$ with probability $1 - \theta$), so the prior probability of category 1 is $\theta$: $P(c_n = 1) = \theta$.

**For all of Problem 2, assume $\mu_1 = -1$ and $\mu_2 = 1$.**


## Part 2(a): Derivation — Categorization

Using **Bayes' rule**, derive the probability of a single datum being in category 1: $P(c_1 = 1 | x_1)$. You can assume that the values of $\mu_1, \mu_2, \sigma_1^2,$ and $\sigma_2^2$ are given parameters. Show your work (handwritten derivation scanned in as an image is fine).

As the next problem depends on this answer, the derivation should end up with:

$$P(c_1 = 1 | x_1) = \frac{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)}$$


#### **Derivation**

*(fill in your derivation here — show your work)*


## Part 2(b): Categorization

Calculate and plot $P(c_1 = 1 | x_1)$ for:

1. $\theta = 0.5$ and $\theta = 0.75$, with $\sigma_1^2 = \sigma_2^2 = 1$.
2. $\theta = 0.5$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.
3. $\theta = 0.75$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.

**Question.** Describe the effect of changing the prior and the variance on categorization decisions. Do they have the same effect? Why or why not?


In [ ]:
# fill me
#
# Plot P(c=1|x) for each configuration.

mu_1, mu_2 = -1.0, 1.0
theta_values = [0.5, 0.75]
configs = [
    (1, 1),      # sigma_1^2 = sigma_2^2 = 1
    (0.5, 2),    # sigma_1^2 = 0.5, sigma_2^2 = 2
]
colors = {0.5: "blue", 0.75: "orange"}
linestyles = {1: "-", 0.5: "--", 2: ":"}

x_range = np.linspace(-6, 6, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

# Suggested approach:
#   1. For each (theta, (sigma_1_sq, sigma_2_sq)) combination:
#      lik1 = norm.pdf(x_range, mu_1, np.sqrt(sigma_1_sq))
#      lik2 = norm.pdf(x_range, mu_2, np.sqrt(sigma_2_sq))
#      posterior_c1 = (theta * lik1) / (theta * lik1 + (1 - theta) * lik2)
#   2. ax.plot(x_range, posterior_c1, label=..., color=colors[theta], linestyle=...)
#   3. Pick label/linestyle so configurations are distinguishable.

# your plotting loop here

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$P(c_1 = 1 | x)$')
ax.set_title('Posterior categorization probability')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Now in GenJAX — Part 2(b): the mixture model + rejection-sampling for $P(c=1|x)$

**Concept (Tutorial 2 Ch 4): conditioning a *discrete* latent.** Here the unknown is $c \in \{1, 2\}$ (category). The posterior $P(c=1|x_1)$ is a discrete probability — one number, not a distribution over $\mu$.

**Two ways to get it in GenJAX:**

1. **`generate` + importance.** Build a choice map `{"x": x_1}`, run `generate` many times, compute $P(c=1|x_1)$ as the importance-weighted fraction of traces where `"category" == 1`.
2. **Rejection sampling.** Simulate many traces unconditionally, keep only those whose `"x"` is close to $x_1$, count what fraction have `"category" == 1`. **Inefficient** because $x_1$ is continuous, but conceptually transparent.

We'll use **approach 1** here. The catch: GenJAX `bernoulli(theta)` returns `True`/`False`, not `1`/`2`. Map `True → c=1, False → c=2`.

**Key syntax preview** (you'll use the same model in Part 2(e)):
- `c = bernoulli(theta) @ "category"` — discrete choice
- `mu_c = jnp.where(c, mu_1, mu_2)` — pick component mean by category (works because c is 0/1)
- `x = normal(mu_c, sigma_c) @ "observation"` — likelihood


In [ ]:
# fill me — optional GenJAX path
#
# Suggested approach:
#   1. Finish the @gen `mixture_model` below — uncomment and fill in.
#   2. For ONE choice of (theta, sigma_1, sigma_2):
#      For each value x_1 in a coarse grid (say 50 points over x_range):
#        - constraints = ChoiceMap({"observation": x_1})
#        - importance-sample N=2000 traces with `generate`
#        - P(c=1|x_1) = weighted mean of trace.get_choices()["category"]
#      Plot empirical P(c=1|x) on top of the analytical curve from the numpy cell above.
#      They should match.
#   3. This is SLOW (2000 traces * 50 grid points = 100k generates). That's fine — the
#      point is to see GenJAX returning the right answer. You can also use a vmap
#      approach (Tutorial 2 Ch 6) to speed it up substantially.

if _GENJAX_AVAILABLE:
    @gen
    def mixture_model(theta, mu_1, mu_2, sigma_1, sigma_2):
        # c = bernoulli(theta) @ "category"       # True (cat 1) or False (cat 2)
        # mu_c = jnp.where(c, mu_1, mu_2)
        # sigma_c = jnp.where(c, sigma_1, sigma_2)
        # x = normal(mu_c, sigma_c) @ "observation"
        # return x, c
        pass

    # Pick ONE configuration, build a grid, and importance-sample at each grid point.
    # your code here
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 2(b)).**

- What is the effect of varying $\theta$ vs. varying the variances?
- Do they have the same effect? Why or why not?

*(your answer here)*


## Part 2(c): Derivation — Prediction

Using Bayes' rule and the **Law of Total Probability**, derive $p(x_1)$ for this model (without any given data). As the next problem depends on this answer, the derivation should end up with:

$$p(x_1) = \theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)$$


#### **Derivation**

*(fill in your derivation here — show your work)*


## Part 2(d): Prediction

Plot $p(x_1)$ for the same configurations as in Part 2(b).

This $p(x_1)$ is sometimes called the *marginal data distribution*; this model is called a **mixture model** because it composes a new distribution by mixing two (or more) component distributions.

**Question.** How does the prior and variance affect $p(x_1)$? Do they have the same effect? Why or why not?


In [ ]:
# fill me
#
# Plot the marginal p(x) for each configuration.

mu_1, mu_2 = -1.0, 1.0
theta_values = [0.5, 0.75]
configs = [
    (1, 1),
    (0.5, 2),
]
colors = {0.5: "blue", 0.75: "orange"}
linestyles = {1: "-", 0.5: "--"}

x_range = np.linspace(-6, 6, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

# Suggested approach:
#   p_x = theta * norm.pdf(x_range, mu_1, sqrt(sigma_1_sq)) +
#         (1 - theta) * norm.pdf(x_range, mu_2, sqrt(sigma_2_sq))

# your plotting loop here

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$p(x)$')
ax.set_title('Marginal (predictive) distribution')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Now in GenJAX — Part 2(d): sampling the marginal $p(x)$

**Concept (Tutorial 2 Ch 2): forward simulation = sampling from the marginal.** If you simulate from the mixture model *without* conditioning on anything, the distribution of observed `x` values **is** the marginal $p(x)$. This is the simplest GenJAX cell in the assignment — no choice map, no importance sampling, just `simulate`.

**Key syntax:**
- `traces = jax.vmap(lambda k: mixture_model.simulate(k, args))(keys)` — N parallel simulations
- `x_samples = traces.get_choices()["observation"]` — pull the observed-x value out of each trace
- `plt.hist(x_samples, bins=80, density=True)` — empirical marginal


In [ ]:
# fill me — optional GenJAX path
#
# Suggested approach:
#   1. Reuse `mixture_model` from Part 2(b). (If you skipped that cell, copy the @gen def
#      from there.)
#   2. For ONE configuration (say theta=0.75, sigma_1=sigma_2=1):
#      - keys = random.split(key, N) with N = 5000.
#      - traces = jax.vmap(lambda k: mixture_model.simulate(k, args))(keys)
#      - x_samples = traces.get_choices()["observation"]
#      - Plot histogram (density=True) of x_samples, overlay the analytical p(x) from above.
#      Verify they match.
#   3. Optional: repeat for the (0.5, 2) variance configuration to see the bimodality emerge.

if _GENJAX_AVAILABLE:
    theta = 0.75
    mu_1, mu_2 = -1.0, 1.0
    sigma_1, sigma_2 = 1.0, 1.0
    N = 5000

    # keys = random.split(key, N)
    # traces = jax.vmap(lambda k: mixture_model.simulate(k, (theta, mu_1, mu_2, sigma_1, sigma_2)))(keys)
    # x_samples = traces.get_choices()["observation"]

    # your histogram + analytical-overlay plotting code here
    pass
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 2(d)).**

- What is the effect of varying $\theta$ vs. the variances on $p(x_1)$?
- Do they have the same effect? Why or why not?

*(your answer here)*


---

## Submission

Submit this completed notebook (runs end-to-end with no errors) plus your derivations for Parts 2(a) and 2(c) (either inline as LaTeX or as scanned/typeset images).

If you completed the optional **"Now in GenJAX"** cells, you've also done the equivalent of the canonical-stencil's Part 2(e) Monte Carlo mixture model. Mention this in your submission for partial bonus consideration.
